In [0]:
from pyspark.sql.functions import *

In [0]:
source_path = "/Volumes/retailnova/bronze/retailnova_source/retailnova_datasets/products/"
target_table = "retailnova.bronze.products"

In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(source_path)

In [0]:
print("Total records found:", df.count())

Total records found: 10600


In [0]:
# Get the last processed timestamp

last_processed = spark.sql("""
    SELECT last_processed_at
    FROM retailnova.bronze.etl_control
    WHERE source_name = 'products'
""").first()[0]

print("Last processed:", last_processed)

Last processed: 2026-08-29 00:00:00


In [0]:
# Keep only records newer than the watermark

new_products = df.filter(
    col("updated_at") > last_processed
)

print("New/changed products:", new_products.count())

New/changed products: 600


In [0]:
# Write new/changed records to Bronze

new_products.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(target_table)

print("Records written to Bronze:", new_products.count())

Records written to Bronze: 600


In [0]:
# Calculate the new watermark

new_watermark = spark.sql("""
    SELECT MAX(updated_at)
    FROM retailnova.bronze.products
""").first()[0]

print("New watermark:", new_watermark)

New watermark: 2026-09-11


In [0]:
# Update the watermark table

spark.sql(f"""
    UPDATE retailnova.bronze.etl_control
    SET last_processed_at = TIMESTAMP('{new_watermark}')
    WHERE source_name = 'products'
""")

print("Watermark table updated successfully")

Watermark table updated successfully


In [0]:
%sql

SELECT COUNT(*)
FROM retailnova.bronze.products;

COUNT(*)
10600


In [0]:
%sql
SELECT *
FROM retailnova.bronze.etl_control;

source_name,last_processed_at
customers_silver,2026-09-10T23:59:10.000Z
customers,2026-09-10T23:59:10.000Z
products_silver,2026-08-29T00:00:00.000Z
products,2026-09-11T00:00:00.000Z
